# Decision Tree in JAX

This notebook demonstrates a supervised classification workflow using a small decision tree built from scratch.

The example uses XOR-style synthetic data so the tree has to learn axis-aligned splits.

## Theory

A decision tree recursively splits the feature space into regions. A common split criterion for classification is Gini impurity:

$$
G = 1 - um_{c} p_c^2
$$

The best split minimizes the weighted impurity of the child nodes.

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

key = jax.random.PRNGKey(1)
x = jax.random.uniform(key, (240, 2), minval=-2.0, maxval=2.0)
y = jnp.logical_xor(x[:, 0] > 0, x[:, 1] > 0).astype(jnp.int32).reshape(-1, 1)

In [ ]:
def gini(labels):
    if labels.shape[0] == 0:
        return 0.0
    p = jnp.mean(labels.astype(jnp.float32))
    return float(1.0 - p ** 2 - (1.0 - p) ** 2)

def split_data(inputs, labels, feature, threshold):
    mask = inputs[:, feature] <= threshold
    return inputs[mask], labels[mask], inputs[~mask], labels[~mask]

def best_split(inputs, labels):
    best = None
    best_score = jnp.inf
    for feature in range(inputs.shape[1]):
        values = jnp.unique(inputs[:, feature])
        thresholds = (values[:-1] + values[1:]) / 2.0
        for threshold in thresholds:
            left_x, left_y, right_x, right_y = split_data(inputs, labels, feature, float(threshold))
            score = (left_y.shape[0] * gini(left_y) + right_y.shape[0] * gini(right_y)) / labels.shape[0]
            if score < best_score:
                best_score = score
                best = (feature, float(threshold), left_x, left_y, right_x, right_y)
    return best

def build_tree(inputs, labels, depth=0, max_depth=2, min_samples=10):
    if depth >= max_depth or labels.shape[0] < min_samples or jnp.all(labels == labels[0]):
        prediction = int(jnp.round(jnp.mean(labels.astype(jnp.float32))))
        return {'type': 'leaf', 'prediction': prediction}
    split = best_split(inputs, labels)
    if split is None:
        prediction = int(jnp.round(jnp.mean(labels.astype(jnp.float32))))
        return {'type': 'leaf', 'prediction': prediction}
    feature, threshold, left_x, left_y, right_x, right_y = split
    return {
        'type': 'node',
        'feature': feature,
        'threshold': threshold,
        'left': build_tree(left_x, left_y, depth + 1, max_depth, min_samples),
        'right': build_tree(right_x, right_y, depth + 1, max_depth, min_samples)
    }

def predict_row(tree, row):
    if tree['type'] == 'leaf':
        return tree['prediction']
    if float(row[tree['feature']]) <= tree['threshold']:
        return predict_row(tree['left'], row)
    return predict_row(tree['right'], row)

def predict(tree, inputs):
    return jnp.array([predict_row(tree, row) for row in inputs]).reshape(-1, 1)

tree = build_tree(x, y)
predictions = predict(tree, x)
accuracy = float(jnp.mean(predictions == y))
print(tree)
print(f'accuracy = {accuracy:.3f}')

## Result

The tree should recover the XOR structure by splitting the feature space into simpler regions.

In [ ]:
grid_x, grid_y = jnp.meshgrid(jnp.linspace(-2.0, 2.0, 200), jnp.linspace(-2.0, 2.0, 200))
grid_points = jnp.stack([grid_x.ravel(), grid_y.ravel()], axis=1)
grid_pred = predict(tree, grid_points).reshape(grid_x.shape)

plt.figure(figsize=(7, 6))
plt.contourf(grid_x, grid_y, grid_pred, levels=2, cmap='coolwarm', alpha=0.4)
plt.scatter(x[:, 0], x[:, 1], c=y[:, 0], cmap='coolwarm', edgecolor='black', s=25)
plt.title('Decision Tree in JAX')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.show()